# 4. Personalisation

Starting from the fold-A model, adds 2, 5 and 10 minutes of one speaker's own recordings and tests on that speaker's unseen sentences. Needs notebook 3 run with `HELDOUT = "A"` first. About 30 minutes on an L4.

In [ ]:
# Setup: run this cell first. On Colab it clones the repo, installs packages and mounts Google Drive.
REPO_URL = "https://github.com/armaan-khoja/clear-voice"
SMOKE = False  # True = a quick check on a handful of clips (minutes, not hours)

import os, subprocess, sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not os.path.exists("/content/clear-voice"):
        subprocess.run(["git", "clone", "-q", REPO_URL, "/content/clear-voice"], check=True)
    os.chdir("/content/clear-voice")
    subprocess.run(["git", "pull", "-q"], check=False)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
    from google.colab import drive, userdata
    drive.mount("/content/drive")
    WORK = "/content/drive/MyDrive/clearvoice"
    try:
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    except Exception:
        print("No HF_TOKEN secret: downloads still work, uploading models will not.")
else:
    WORK = os.path.abspath("work")
sys.path.insert(0, os.getcwd())
os.makedirs(f"{WORK}/results", exist_ok=True)
os.makedirs(f"{WORK}/models", exist_ok=True)
SPLITS = "splits"
LIMIT = 6 if SMOKE else None
print("Results and models go to", WORK)

In [ ]:
import torch
print(torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else "No GPU: Runtime > Change runtime type > L4 GPU, then rerun the setup cell.")

In [ ]:
SPEAKERS = ["M01", "F03"]   # stretch: add "M03" (fold A) and "M04" (needs the fold-B model)
MINUTES = [2, 5, 10]
FOLD_OF = {"M01": "A", "F03": "A", "M03": "A", "M04": "B"}

In [ ]:
from clearvoice.pipeline import load_torgo
parquets, meta = load_torgo(WORK)  # downloads TORGO (about 1.6 GB) the first time in each session
print(len(meta), "clips from", meta.speaker.nunique(), "speakers")

In [ ]:
import os, shutil, tempfile
import pandas as pd, torch
from clearvoice.asr import load_model
from clearvoice.data import load_audio
from clearvoice.metrics import score
from clearvoice.pipeline import predict, rows_for
from clearvoice.train import TrainConfig, finetune

def evaluate(model_dir, test, test_audio, fold, minutes):
    model, processor = load_model(model_dir)
    p = predict(model, processor, parquets, meta, test, "Clear Voice", fold, "test_strict", audio=test_audio)
    s = score(list(p.ref), list(p.hyp))
    del model
    torch.cuda.empty_cache()
    return {"speaker": test.speaker.iloc[0], "severity": test.severity.iloc[0], "minutes": minutes,
            "wer": s["wer"], "cer": s["cer"], "n_clips": s["n"]}

results = []
for spk in SPEAKERS:
    fold = FOLD_OF[spk]
    base = f"{WORK}/models/hold{fold}/best"
    test = rows_for(meta, SPLITS, f"hold{fold}", "test_strict")
    test = test[test.speaker == spk].head(LIMIT or len(test))
    test_audio = load_audio(parquets, meta, test.row_id)
    results.append(evaluate(base, test, test_audio, fold, 0))
    for k in MINUTES:
        path = f"{SPLITS}/personal_{spk}_{k}min.csv"
        if not os.path.exists(path):
            print(f"{spk}: {k} min is more than this speaker has (capped)")
            continue
        adapt = rows_for(meta, SPLITS, f"personal_{spk}", f"{k}min")
        adapt = adapt.head(8) if SMOKE else adapt
        audio = load_audio(parquets, meta, adapt.row_id)
        cfg = TrainConfig(base_model=base, max_steps=2 if SMOKE else None, num_train_epochs=5, eval=False, batch_size=8)
        tmp = tempfile.mkdtemp()  # personal models stay off Google Drive (about 1 GB each)
        out = finetune(cfg, adapt, None, audio, tmp)
        results.append(evaluate(out, test, test_audio, fold, k))
        shutil.rmtree(tmp, ignore_errors=True)
        print(spk, k, "min done")

res = pd.DataFrame(results)
res.to_csv(f"{WORK}/results/personalise.csv", index=False)
res